# TP 7 — Machine Learning avec Python : Clustering avec K-Means (Sklearn)

---
**Université Internationale à Laâyoune — UNIVERSIAPOLIS**  
**Filière :** Génie Informatique &nbsp;|&nbsp; **Niveau :** 4ème Année  
**Matière :** Sciences des Données Appliquées &nbsp;|&nbsp; **Session :** 2/2026

---

## 📦 Importation des bibliothèques

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

print('✅ Bibliothèques importées avec succès !')

---
# 🏋️ Exercice 1 : Sahara Technologie — Segmentation en 3 clusters

### Étape 1 — Représenter les données sous forme de tableau (pandas)

In [ ]:
données = {
    'Client': ['Ahmed (A)', 'Ali (B)', 'Mariem (C)', 'Hamma (D)',
               'Brahim (E)', 'Laila (F)', 'Ghlana (G)', 'Salma (H)',
               'Malaynine (I)', 'Saleh (J)', 'Salma (K)',
               'Malaynine (L)', 'Saleh (M)'],
    'Mardi (DH)':    [4000, 2000, 8000, 3000, 7000, 400, 1500, 9000, 700, 300, 6500, 1700, 5000],
    'Vendredi (DH)': [6000, 1000,  300, 8000, 9000, 800, 6500, 9500, 500, 300, 9500,  550, 3000],
}

df = pd.DataFrame(données).set_index('Client')
print('📊 Tableau des transferts clients :')
df

### Étape 2 — Visualiser les clients sur un plan 2D (Mardi vs Vendredi)

In [ ]:
X = df[['Mardi (DH)', 'Vendredi (DH)']].values
labels_clients = df.index.tolist()

plt.figure(figsize=(9, 6))
plt.scatter(X[:, 0], X[:, 1], color='steelblue', s=120,
            edgecolors='white', linewidths=1.5, zorder=3)

for i, nom in enumerate(labels_clients):
    plt.annotate(nom, (X[i, 0], X[i, 1]),
                 textcoords='offset points', xytext=(8, 6), fontsize=8)

plt.title('Visualisation des clients (Mardi vs Vendredi)',
          fontsize=14, fontweight='bold')
plt.xlabel('Transferts Mardi (DH)')
plt.ylabel('Transferts Vendredi (DH)')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

### Étape 3 — Appliquer K-Means (K = 3)

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X)

étiquettes = kmeans.labels_
centroïdes = kmeans.cluster_centers_

print(f'Inertie (WCSS)     : {kmeans.inertia_:.2f}')
print(f'Score Silhouette   : {silhouette_score(X, étiquettes):.4f}')

### Étapes 4 & 5 — Afficher les clusters et ajouter la colonne Cluster

In [ ]:
couleurs = ['#E63946', '#2A9D8F', '#F4A261']

plt.figure(figsize=(10, 7))
for k in range(3):
    masque = étiquettes == k
    plt.scatter(X[masque, 0], X[masque, 1], color=couleurs[k], s=130,
                edgecolors='white', linewidths=1.5, label=f'Cluster {k}', zorder=3)

plt.scatter(centroïdes[:, 0], centroïdes[:, 1],
            marker='X', s=280, color='black', zorder=5, label='Centroïdes')

for i, nom in enumerate(labels_clients):
    plt.annotate(nom, (X[i, 0], X[i, 1]),
                 textcoords='offset points', xytext=(8, 6), fontsize=7.5)

plt.title('K-Means (K=3) — Sahara Technologie', fontsize=13, fontweight='bold')
plt.xlabel('Mardi (DH)'); plt.ylabel('Vendredi (DH)')
plt.legend(); plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout(); plt.show()

# Ajouter la colonne Cluster
df['Cluster'] = étiquettes
df['Groupe']  = df['Cluster'].map({0: 'Cluster A', 1: 'Cluster B', 2: 'Cluster C'})
print('\n📊 Tableau avec Cluster :')
df

### Étape 6 — Interprétation des clusters

In [ ]:
for k in range(3):
    sous = df[df['Cluster'] == k]
    print(f'\n🔵 Cluster {k} ({len(sous)} clients) :')
    print(f'   Membres : {", ".join(sous.index)}')
    print(f'   Moy. Mardi    : {sous["Mardi (DH)"].mean():,.0f} DH')
    print(f'   Moy. Vendredi : {sous["Vendredi (DH)"].mean():,.0f} DH')

### Étape 7 — Stratégie de bonus par cluster

| Cluster | Profil | Bonus |
|---------|--------|-------|
| Cluster à faibles transferts | Clients peu actifs | **100 %** |
| Cluster à transferts moyens | Clients réguliers | **150 %** |
| Cluster à forts transferts | Clients premium | **200 %** |

### Étape 8 — Comparaison K=2 et K=4

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
palette = plt.cm.tab10.colors

for idx, k_val in enumerate([2, 4]):
    km = KMeans(n_clusters=k_val, random_state=42, n_init=10)
    km.fit(X)
    etiq = km.labels_
    ctrs = km.cluster_centers_
    sil  = silhouette_score(X, etiq)
    ax   = axes[idx]

    for k in range(k_val):
        masque = etiq == k
        ax.scatter(X[masque, 0], X[masque, 1], color=palette[k], s=120,
                   edgecolors='white', linewidths=1.5, label=f'Cluster {k+1}', zorder=3)

    ax.scatter(ctrs[:, 0], ctrs[:, 1], marker='X', s=260, color='black',
               zorder=5, label='Centroïdes')

    for i, nom in enumerate(labels_clients):
        ax.annotate(nom, (X[i, 0], X[i, 1]),
                    textcoords='offset points', xytext=(5, 5), fontsize=7)

    ax.set_title(f'K-Means K={k_val} | Silhouette = {sil:.4f}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Mardi (DH)'); ax.set_ylabel('Vendredi (DH)')
    ax.legend(fontsize=8); ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Comparaison K=2 et K=4', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### Étape 9 — Méthode du coude et silhouette

In [ ]:
inertie_lst   = []
silhouette_lst = []
K_range = range(2, 8)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertie_lst.append(km.inertia_)
    silhouette_lst.append(silhouette_score(X, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(list(K_range), inertie_lst, marker='o', color='#E63946', linewidth=2.5)
ax1.axvline(x=3, color='gray', linestyle='--', alpha=0.7, label='K=3 optimal')
ax1.set_title('Méthode du Coude', fontsize=13, fontweight='bold')
ax1.set_xlabel('K'); ax1.set_ylabel('Inertie')
ax1.legend(); ax1.grid(True, linestyle='--', alpha=0.4)

ax2.plot(list(K_range), silhouette_lst, marker='s', color='#2A9D8F', linewidth=2.5)
meilleur_k = list(K_range)[silhouette_lst.index(max(silhouette_lst))]
ax2.axvline(x=meilleur_k, color='gray', linestyle='--', alpha=0.7,
            label=f'K={meilleur_k} optimal')
ax2.set_title('Score Silhouette', fontsize=13, fontweight='bold')
ax2.set_xlabel('K'); ax2.set_ylabel('Silhouette')
ax2.legend(); ax2.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Choix automatique du meilleur K', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Meilleur K : {meilleur_k}')

### Étapes 10 & 11 — Prédire les nouveaux clients

In [ ]:
nouveaux = pd.DataFrame({
    'Client':        ['Ahmed (N)', 'Ali (O)', 'Mariem (P)', 'Hamma (Q)'],
    'Mardi (DH)':    [4400, 3000, 700, 1000],
    'Vendredi (DH)': [5000, 1500, 1300, 9200],
}).set_index('Client')

X_nouveaux  = nouveaux[['Mardi (DH)', 'Vendredi (DH)']].values
predictions = kmeans.predict(X_nouveaux)
nouveaux['Cluster'] = predictions
nouveaux['Groupe']  = nouveaux['Cluster'].map(
    {0: 'Cluster A', 1: 'Cluster B', 2: 'Cluster C'})

print('📊 Prédictions pour les nouveaux clients :')
nouveaux

---
# 🏬 Exercice 2 : Supermarché — Mall Customers (K-Means classique)

In [ ]:
# Chargement ou génération du dataset Mall Customers
try:
    df_mall = pd.read_csv('Mall_Customers.csv')
    df_mall = df_mall.rename(columns={
        'Annual Income (k$)':     'Revenu_Annuel_k$',
        'Spending Score (1-100)': 'Score_Depense',
        'Gender':                 'Genre',
    })
    print('✅ Mall_Customers.csv chargé.')
except FileNotFoundError:
    np.random.seed(42)
    n = 200
    groupes = [
        np.random.multivariate_normal([30,30,20], np.diag([50,50,100]), 40),
        np.random.multivariate_normal([55,55,49], np.diag([60,60,200]), 40),
        np.random.multivariate_normal([45,86,82], np.diag([40,40,150]), 40),
        np.random.multivariate_normal([40,26,77], np.diag([50,50,150]), 40),
        np.random.multivariate_normal([32,87,17], np.diag([50,50,150]), 40),
    ]
    data = np.vstack(groupes)
    df_mall = pd.DataFrame({
        'CustomerID':       range(1, n+1),
        'Genre':            np.random.choice(['Male','Female'], n),
        'Age':              np.clip(data[:,0], 18, 70).astype(int),
        'Revenu_Annuel_k$': np.clip(data[:,1], 15, 137).astype(int),
        'Score_Depense':    np.clip(data[:,2], 1,  99).astype(int),
    })
    print('⚠️  Dataset simulé (200 clients).')

print(f'Dimensions : {df_mall.shape}')
df_mall.head()

In [ ]:
# Normalisation
features_mall = ['Age', 'Revenu_Annuel_k$', 'Score_Depense']
features_mall = [f for f in features_mall if f in df_mall.columns]

scaler_mall = StandardScaler()
X_mall_norm = scaler_mall.fit_transform(df_mall[features_mall])
print('✅ Normalisation effectuée.')

# Coude + Silhouette
inertie_m, sil_m = [], []
K_r = range(2, 11)
for k in K_r:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_mall_norm)
    inertie_m.append(km.inertia_)
    sil_m.append(silhouette_score(X_mall_norm, km.labels_))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
a1.plot(list(K_r), inertie_m, 'o-', color='#E63946', lw=2.5)
a1.set_title('Méthode du Coude — Mall', fontsize=12, fontweight='bold')
a1.set_xlabel('K'); a1.set_ylabel('Inertie'); a1.grid(True, ls='--', alpha=0.4)
a2.plot(list(K_r), sil_m, 's-', color='#2A9D8F', lw=2.5)
best_k_mall = list(K_r)[sil_m.index(max(sil_m))]
a2.axvline(x=best_k_mall, color='gray', ls='--', alpha=0.7, label=f'K={best_k_mall}')
a2.set_title('Silhouette — Mall', fontsize=12, fontweight='bold')
a2.set_xlabel('K'); a2.set_ylabel('Silhouette'); a2.legend(); a2.grid(True, ls='--', alpha=0.4)
plt.tight_layout(); plt.show()
print(f'Meilleur K : {best_k_mall}')

In [ ]:
# Application K-Means avec K=5
km5 = KMeans(n_clusters=5, random_state=42, n_init=10)
km5.fit(X_mall_norm)
df_mall['Cluster'] = km5.labels_

print(f'Inertie : {km5.inertia_:.2f}')
print(f'Silhouette : {silhouette_score(X_mall_norm, km5.labels_):.4f}\n')

profil_mall = df_mall.groupby('Cluster')[features_mall].mean().round(1)
profil_mall['Effectif'] = df_mall['Cluster'].value_counts().sort_index()
print('📊 Profil moyen par cluster :')
profil_mall

In [ ]:
# Visualisation des clusters
couleurs_mall = ['#E63946','#2A9D8F','#F4A261','#457B9D','#A8DADC']

plt.figure(figsize=(9, 6))
for k in range(5):
    m = df_mall['Cluster'] == k
    plt.scatter(df_mall.loc[m,'Revenu_Annuel_k$'],
                df_mall.loc[m,'Score_Depense'],
                color=couleurs_mall[k], s=70, alpha=0.8,
                label=f'Cluster {k}', edgecolors='white', lw=0.5)

ctrs_orig = scaler_mall.inverse_transform(km5.cluster_centers_)
rev_idx   = features_mall.index('Revenu_Annuel_k$')
dep_idx   = features_mall.index('Score_Depense')
plt.scatter(ctrs_orig[:, rev_idx], ctrs_orig[:, dep_idx],
            marker='X', s=250, color='black', zorder=5, label='Centroïdes')

plt.title('Revenu vs Score de Dépense (K-Means K=5)',
          fontsize=13, fontweight='bold')
plt.xlabel('Revenu Annuel (k$)'); plt.ylabel('Score de Dépense')
plt.legend(fontsize=9); plt.grid(True, ls='--', alpha=0.4)
plt.tight_layout(); plt.show()

---
# 🏢 Exercice 3 : Centre Commercial Marjane — Analyse complète

In [ ]:
# Génération ou chargement du dataset Marjane
np.random.seed(2024)
N = 200

try:
    df_marjane = pd.read_csv('dataset_marjane_clients.csv')
    print('✅ dataset_marjane_clients.csv chargé.')
except FileNotFoundError:
    data_list = []
    for i in range(1, N+1):
        a_telephone = np.random.random() > 0.4
        a_carte     = np.random.random() > 0.5 if a_telephone else False
        achat       = np.random.randint(500, 25000)
        data_list.append({
            'client_id':         i,
            'telephone':         f'06{np.random.randint(10000000,99999999)}' if a_telephone else None,
            'carte_fidelite':    f'FID{np.random.randint(10000,99999)}' if a_carte else None,
            'achat_total_dh':    achat,
            'nombre_visites':    np.random.randint(1, 60),
            'solde_fidelite_dh': round(achat * 0.004 * np.random.uniform(0.5,1.5), 2),
            'moyen_paiement':    np.random.choice(['Cash','Mobile','Carte Bancaire'], p=[0.4,0.35,0.25]),
            'categorie_achat':   np.random.choice(['Alimentaire','Electronique','Vêtements','Maison'], p=[0.35,0.25,0.2,0.2]),
            'heure_achat':       np.random.randint(8, 22),
            'client_enregistre': 1 if a_carte else 0,
        })
    df_marjane = pd.DataFrame(data_list)
    df_marjane.to_csv('dataset_marjane_clients.csv', index=False)
    print('⚠️  Dataset simulé créé.')

print(f'Dimensions : {df_marjane.shape}')
df_marjane.head()

In [ ]:
# Préparation et normalisation
features_marjane = ['achat_total_dh', 'nombre_visites',
                    'solde_fidelite_dh', 'client_enregistre']

scaler_mj = StandardScaler()
X_mj_norm = scaler_mj.fit_transform(df_marjane[features_marjane])
print('✅ Normalisation effectuée.')
print(f'Valeurs manquantes :\n{df_marjane.isnull().sum()}')

In [ ]:
# Méthode du coude + silhouette
inertie_mj, sil_mj = [], []
K_r = range(2, 11)
for k in K_r:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_mj_norm)
    inertie_mj.append(km.inertia_)
    sil_mj.append(silhouette_score(X_mj_norm, km.labels_))

fig, (a1,a2) = plt.subplots(1,2,figsize=(13,5))
a1.plot(list(K_r), inertie_mj,'o-',color='#E63946',lw=2.5)
a1.set_title('Méthode du Coude — Marjane',fontsize=12,fontweight='bold')
a1.set_xlabel('K'); a1.set_ylabel('Inertie'); a1.grid(True,ls='--',alpha=0.4)
a2.plot(list(K_r), sil_mj,'s-',color='#2A9D8F',lw=2.5)
best_mj = list(K_r)[sil_mj.index(max(sil_mj))]
a2.axvline(x=best_mj,color='gray',ls='--',alpha=0.7,label=f'K={best_mj}')
a2.set_title('Silhouette — Marjane',fontsize=12,fontweight='bold')
a2.set_xlabel('K'); a2.set_ylabel('Silhouette'); a2.legend(); a2.grid(True,ls='--',alpha=0.4)
plt.suptitle('Choix du meilleur K — Marjane',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Meilleur K : {best_mj}')

In [ ]:
# Application K-Means K=4
km_mj = KMeans(n_clusters=4, random_state=42, n_init=10)
km_mj.fit(X_mj_norm)
df_marjane['Cluster'] = km_mj.labels_

profil_mj = df_marjane.groupby('Cluster')[features_marjane].mean().round(2)
profil_mj['Effectif'] = df_marjane['Cluster'].value_counts().sort_index()
print(f'Inertie : {km_mj.inertia_:.2f}')
print(f'Silhouette : {silhouette_score(X_mj_norm, km_mj.labels_):.4f}\n')
print('📊 Profil moyen par cluster :')
profil_mj

In [ ]:
# Visualisation clusters Marjane
couleurs_mj = ['#E63946','#2A9D8F','#F4A261','#457B9D']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for k in range(4):
    m = df_marjane['Cluster'] == k
    axes[0].scatter(df_marjane.loc[m,'achat_total_dh'],
                    df_marjane.loc[m,'nombre_visites'],
                    color=couleurs_mj[k], s=55, alpha=0.75,
                    label=f'Cluster {k}', edgecolors='white', lw=0.4)
    axes[1].scatter(df_marjane.loc[m,'achat_total_dh'],
                    df_marjane.loc[m,'solde_fidelite_dh'],
                    color=couleurs_mj[k], s=55, alpha=0.75,
                    label=f'Cluster {k}', edgecolors='white', lw=0.4)

for ax, y_label, title in zip(axes,
    ['Nombre de Visites','Solde Fidélité (DH)'],
    ['Achat Total vs Visites','Achat Total vs Solde Fidélité']):
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Achat Total (DH)'); ax.set_ylabel(y_label)
    ax.legend(fontsize=9); ax.grid(True, ls='--', alpha=0.4)

plt.suptitle('Clusters K-Means — Clients Marjane (K=4)',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Heatmap de corrélation
plt.figure(figsize=(7, 5))
corr = df_marjane[features_marjane].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True, vmin=-1, vmax=1)
plt.title('Heatmap de Corrélation — Marjane',
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout(); plt.show()

In [ ]:
# Comparaison K-Means vs DBSCAN
dbscan = DBSCAN(eps=0.6, min_samples=5)
labels_db = dbscan.fit_predict(X_mj_norm)

n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_bruit       = list(labels_db).count(-1)

print(f'DBSCAN — Clusters : {n_clusters_db} | Points bruit : {n_bruit}')
print(f'K-Means — Silhouette : {silhouette_score(X_mj_norm, km_mj.labels_):.4f}')

if n_clusters_db > 1:
    masque_valid = labels_db != -1
    print(f'DBSCAN  — Silhouette (sans bruit) : {silhouette_score(X_mj_norm[masque_valid], labels_db[masque_valid]):.4f}')

---
## ✅ Conclusion générale

| Aspect | Résultat |
|--------|----------|
| Algorithme utilisé | K-Means (sklearn) |
| Normalisation | StandardScaler |
| Sélection du K | Méthode du coude + score silhouette |
| Évaluation | Inertie (WCSS) + silhouette score |
| Comparaison | K-Means vs DBSCAN |

**K-Means** est simple, rapide et efficace pour des données sphériques.  
**DBSCAN** est préférable pour détecter des formes arbitraires et des outliers.